# Stop Paying GPT to Sort Your Email: Let's Build a No-LLM Inbox Assistant (Gmail + Jev + Composio)

**LLMs generate, Jev decides.** Sorting email is a *decision*, not a writing job. So in this notebook the part that replaces the LLM is **[Jev](https://docs.typesafe.ai/sdk/python)** (TypeSafe System One), a decision model that only answers typed questions: pick one option, yes/no, or a score, each with a confidence. Jev never writes text.

**What you'll learn**

1. How Python turns a raw email into a few simple facts (sender, subject, snippet, "has an unsubscribe link"…).
2. How to ask Jev three questions per email in one call: *which category?*, *what action?* and *is this clear enough to sort?*
3. How a confidence gate sends unsure emails to a **Review** label instead of guessing.
4. How to apply Gmail labels and archive with Composio, safely, with `DRY_RUN=True` by default.

**The analogy:** think of a mail room. Python is the clerk who reads the envelope and notes the facts. Jev is the sorter who drops each letter into the right tray: *Urgent*, *Reply later*, *Newsletter*, *Spam*. If the sorter isn't sure, the letter goes to a *Review* tray for you. Composio is the hands that put the tray labels on in Gmail. Nobody in this mail room writes letters.

**Why Composio?** Only for easy Gmail access (one OAuth link, no Google Cloud setup). It is optional. If you prefer, call the Gmail API directly; the swap is shown near the end. The part that replaces the LLM is Jev.

**Safety rules**
- `DRY_RUN=True` by default: nothing is labeled or archived until you flip it.
- The code never deletes or trashes email. "Archive" only removes the `INBOX` label, and the email stays in *All Mail*.
- The notebook runs on a **fictional demo inbox** by default, so anyone can run it. Live Gmail is opt-in.

**Secrets** (Colab: key icon in the left sidebar, enable notebook access). Never paste keys into a cell and never print them.
- `TYPESAFE_API_KEY`: Jev
- `COMPOSIO_API_KEY`: Gmail access (only needed for live Gmail)


In [1]:
%pip install -q composio typesafe-sdk pandas


composio, typesafe-sdk, and pandas are installed.


## Setup: keys, knobs and clients

The knobs at the top are the only things you need to edit.


In [2]:
import os, re, time
from email.utils import parseaddr
from pathlib import Path

import pandas as pd
from typesafe_sdk import Choice, Noul, TypeSafeClient

try:
    from google.colab import userdata
except Exception:
    userdata = None


def get_secret(name, required=True):
    val = None
    if userdata is not None:
        try:
            val = userdata.get(name)
        except Exception:
            val = None
    val = val or os.environ.get(name)
    if not val and required:
        raise RuntimeError(f"Add {name} in Colab Secrets and enable notebook access.")
    if val:
        os.environ[name] = val
    return val


def flag(name, default):
    raw = (os.environ.get(name) or "").strip().lower()
    return {"1": True, "true": True, "yes": True, "0": False, "false": False, "no": False}.get(raw, default)


# --- knobs ---
DRY_RUN = flag("INBOX_DRY_RUN", True)          # True = plan only, never touch Gmail
USE_LIVE_GMAIL = flag("INBOX_LIVE", False)      # False = fictional demo inbox
CONF_MIN = 0.70                                 # category confidence below this -> Review
CLEAR_MIN = 0.50                                # "clear enough?" below this -> Review
MAX_EMAILS = 15                                 # live mode: how many unread emails to read

LABELS = {
    "urgent": "Jev/Urgent",
    "reply_later": "Jev/Reply Later",
    "newsletter": "Jev/Newsletter",
    "spam": "Jev/Spam",
    "review": "Jev/Review",
}

get_secret("TYPESAFE_API_KEY")
jev = TypeSafeClient()          # reads TYPESAFE_API_KEY from the environment
print("Jev client ready. Keys loaded (values not shown).")
print(f"DRY_RUN={DRY_RUN}  USE_LIVE_GMAIL={USE_LIVE_GMAIL}  CONF_MIN={CONF_MIN}  CLEAR_MIN={CLEAR_MIN}")


Jev client ready. Keys loaded (values not shown).
DRY_RUN=True  USE_LIVE_GMAIL=False  CONF_MIN=0.7  CLEAR_MIN=0.5


## The demo inbox

Eighteen fictional emails from fictional senders (every domain ends in `.example`). Each has a hand label in `expected`, so we can check Jev's answers afterwards. Two are deliberately vague; the right answer for those is *"ask a human"* (`review`).


In [3]:
def email(i, sender, subject, snippet, unsubscribe=False, expected=""):
    return {"id": f"demo-{i:02d}", "from": sender, "subject": subject, "snippet": snippet,
            "list_unsubscribe": unsubscribe, "expected": expected}

DEMO_INBOX = [
    email(1, "Ada Mensah <ada@northwind.example>", "Checkout is down for all customers",
          "Payments fail with error 500 since 9:10. Customers are complaining. Can you look right now?", expected="urgent"),
    email(2, "Billing Team <billing@cloudhost.example>", "Final notice: your server will be suspended in 24 hours",
          "Your invoice is 14 days overdue. Pay today to avoid suspension of your account.", expected="urgent"),
    email(3, "Tunde Bello <tunde@acme-studio.example>", "Need the signed contract before 5pm today",
          "The client won't start without it. Please sign and send it back by 5pm.", expected="urgent"),
    email(4, "Building Manager <office@greenview-homes.example>", "Water leak in your flat, access needed this afternoon",
          "Water is coming through the ceiling below you. A plumber needs access today at 2pm. Please confirm.", expected="urgent"),
    email(5, "Chioma Eze <chioma@friends.example>", "Dinner next month?",
          "Hey! Would you be free for dinner sometime next month? No rush, let me know.", expected="reply_later"),
    email(6, "Grace Lee <grace@acme-studio.example>", "Feedback on the onboarding doc",
          "I left a few comments on the onboarding doc. Could you take a look when you get a chance this week?", expected="reply_later"),
    email(7, "Sam Carter <sam@talentbridge.example>", "Intro chat about a role?",
          "I came across your work and would love a short call in the next couple of weeks. Are you open to it?", expected="reply_later"),
    email(8, "Maria Rossi <maria@bluefin-retail.example>", "Question about next quarter's plan",
          "When you have time, could you share your thoughts on the Q1 roadmap? No deadline yet.", expected="reply_later"),
    email(9, "The Weekly Byte <news@weeklybyte.example>", "This week in AI: 7 things you missed",
          "Top stories, new tools and a tutorial of the week. Read in your browser.", unsubscribe=True, expected="newsletter"),
    email(10, "Brightwear <deals@brightwear.example>", "20% off everything this weekend",
          "Our biggest sale of the season starts now. Shop new arrivals.", unsubscribe=True, expected="newsletter"),
    email(11, "DevTalks <events@devtalks.example>", "Webinar invite: scaling Python services",
          "Join our free live session on Thursday. Save your seat.", unsubscribe=True, expected="newsletter"),
    email(12, "Pod Weekly <hello@podweekly.example>", "New episode: building in public",
          "Episode 42 is out now. Listen on your favorite app.", unsubscribe=True, expected="newsletter"),
    email(13, "Code Digest <digest@codedigest.example>", "Your monthly reading list",
          "Ten hand-picked articles about software design this month.", unsubscribe=True, expected="newsletter"),
    email(14, "Prize Center <winner@lucky-draw.example>", "CONGRATULATIONS!!! You have won $5,000,000",
          "Claim your prize now by sending a small processing fee and your bank details.", expected="spam"),
    email(15, "Crypto Max <profit@crypto-max.example>", "Double your Bitcoin in 24 hours, guaranteed",
          "Send any amount and receive double back. Limited spots. Act fast.", expected="spam"),
    email(16, "Secure Bank <alert@secure-bank-verify.example>", "Your account is locked, verify now",
          "Unusual activity detected. Click this link and enter your password to unlock your account.", expected="spam"),
    email(17, "Unknown <j.k@mailbox.example>", "Quick question",
          "Hi, about the thing we discussed. Let me know.", expected="review"),
    email(18, "Leo <leo@random-mail.example>", "Re: that",
          "ok see attached. thoughts?", expected="review"),
]
print(f"Demo inbox: {len(DEMO_INBOX)} fictional emails")
print(pd.Series([e["expected"] for e in DEMO_INBOX]).value_counts().to_string())


Demo inbox: 18 fictional emails
newsletter     5
urgent         4
reply_later    4
spam           3
review         2


## Step 1: Python reads the envelope

No AI here. Plain Python turns each email into a few facts Jev can use: the sender's domain, whether the email has an unsubscribe header (a strong newsletter signal), whether the sender is in your contacts, and whether the text mentions a deadline. This is cheap, fast and easy to test.


In [4]:
MY_CONTACT_DOMAINS = {"northwind.example", "acme-studio.example", "friends.example",
                      "bluefin-retail.example", "greenview-homes.example"}
DEADLINE_WORDS = re.compile(r"\b(today|tonight|now|asap|urgent|by \d{1,2}(:\d\d)?\s?(am|pm)|24 hours|immediately)\b", re.I)


def preprocess(msg):
    name, address = parseaddr(msg["from"])
    domain = address.split("@")[-1].lower() if "@" in address else ""
    text = f"{msg['subject']} {msg['snippet']}"
    return {
        "id": msg["id"],
        "sender_name": name,
        "sender_domain": domain,
        "subject": msg["subject"],
        "snippet": msg["snippet"][:300],
        "has_unsubscribe_header": bool(msg.get("list_unsubscribe")),
        "sender_is_contact": domain in MY_CONTACT_DOMAINS,
        "mentions_deadline": bool(DEADLINE_WORDS.search(text)),
        "asks_for_money_or_password": bool(re.search(r"\b(fee|password|bank details|send any amount)\b", text, re.I)),
    }


facts = [preprocess(m) for m in DEMO_INBOX]
pd.DataFrame(facts)[["id", "sender_domain", "has_unsubscribe_header", "sender_is_contact",
                     "mentions_deadline", "asks_for_money_or_password"]]


,id,sender_domain,has_unsubscribe_header,sender_is_contact,mentions_deadline,asks_for_money_or_password
0,demo-01,northwind.example,False,True,True,False
1,demo-02,cloudhost.example,False,False,True,False
2,demo-03,acme-studio.example,False,True,True,False
3,demo-04,greenview-homes.example,False,True,True,False
4,demo-05,friends.example,False,True,False,False
5,demo-06,acme-studio.example,False,True,False,False
6,demo-07,talentbridge.example,False,False,False,False
7,demo-08,bluefin-retail.example,False,True,False,False
8,demo-09,weeklybyte.example,True,False,False,False
9,demo-10,brightwear.example,True,False,True,False


## Step 2: The three questions we ask Jev

This is the part that replaces the LLM. Instead of a prompt like *"You are a helpful email assistant…"* and hoping the reply parses, we ask Jev **typed questions** with fixed answers:

- **category**: `urgent`, `reply_later`, `newsletter` or `spam`
- **action**: `label` (label it, keep it in the inbox), `archive` (label it, move it out of the inbox) or `leave` (don't touch it)
- **clear**: yes/no, *is there enough information to sort this confidently?* (a probability from 0 to 1)

All three questions go in **one** `system_one` call, and every answer comes back with a confidence.


In [5]:
CATEGORY_Q = Choice(
    instructions="Which inbox category does this email belong in?",
    criteria={
        "urgent": "Needs the reader's action today: outages, hard deadlines today, final notices, emergencies.",
        "reply_later": "A real person expects a reply, but it can wait a few days.",
        "newsletter": "A newsletter, promotion, event invite or other bulk mail the reader signed up for.",
        "spam": "Unwanted junk, scams, phishing, or requests for money or passwords.",
    },
)
ACTION_Q = Choice(
    instructions="What should happen to this email in the Gmail inbox?",
    criteria={
        "label": "Keep it in the inbox with a label: a person needs to read or answer it.",
        "archive": "Label it and move it out of the inbox: newsletters, promotions, event invites, scams and other mail nobody needs to read now.",
        "leave": "Leave it exactly as it is: there is not enough information to decide.",
    },
)
CLEAR_Q = Noul(
    instructions="Is there enough information in this email to sort it confidently?",
    criteria={
        "true": "The sender, subject and text make the purpose clear.",
        "false": "The email is vague or cryptic; a human should look at it.",
    },
)
QUESTIONS = {"category": CATEGORY_Q, "action": ACTION_Q, "clear": CLEAR_Q}


def ask_jev(f):
    state = {k: f[k] for k in ("sender_name", "sender_domain", "subject", "snippet", "has_unsubscribe_header",
                               "sender_is_contact", "mentions_deadline", "asks_for_money_or_password")}
    t0 = time.perf_counter()
    r = jev.system_one(state=state, questions=QUESTIONS)
    ms = (time.perf_counter() - t0) * 1000
    return r, ms


# Try it on one email first
r, ms = ask_jev(facts[0])
print("Email:", facts[0]["subject"])
print("category ->", r.answers["category"].choice, f"(confidence {r.answers['category'].confidence:.2f})")
print("action   ->", r.answers["action"].choice, f"(confidence {r.answers['action'].confidence:.2f})")
print("clear?   ->", f"{r.answers['clear'].noul:.2f}  (1.0 = clearly yes)")
print(f"model={r.model}  time={ms:.0f} ms  tokens in/out={r.usage.input_tokens}/{r.usage.output_tokens}")


Email: Checkout is down for all customers
category -> urgent (confidence 1.00)
action   -> label (confidence 1.00)
clear?   -> 0.91  (1.0 = clearly yes)
model=jev-1.14.0  time=178 ms  tokens in/out=686/99


## Step 3: Sort the whole inbox, with a confidence gate

For every email we ask the same three questions. Then a few lines of Python make it safe:

- If Jev's category confidence is below `CONF_MIN`, or Jev says the email is not clear enough (below `CLEAR_MIN`), it goes to **Jev/Review** and stays in the inbox.
- `urgent` and `reply_later` are **never** archived, whatever the action says.
- Nothing is ever deleted.


In [6]:
def decide(f):
    r, ms = ask_jev(f)
    cat, act, clear = r.answers["category"], r.answers["action"], float(r.answers["clear"].noul)
    category, conf, action = cat.choice, float(cat.confidence), act.choice
    if conf < CONF_MIN or clear < CLEAR_MIN:
        final, label, archive = "review", LABELS["review"], False
    else:
        final, label = category, LABELS[category]
        archive = action == "archive" and category in ("newsletter", "spam")
        if action == "leave" and category in ("newsletter", "spam"):
            label = None  # Jev said don't touch it
    return {"id": f["id"], "subject": f["subject"], "jev_category": category, "confidence": round(conf, 2),
            "jev_action": act.choice, "clear": round(clear, 2), "final": final, "label": label or "-", "archive": archive,
            "ms": round(ms), "tok_in": r.usage.input_tokens, "tok_out": r.usage.output_tokens}


decisions = [decide(f) for f in facts]
table = pd.DataFrame(decisions)
table[["id", "subject", "jev_category", "confidence", "jev_action", "clear", "final", "label", "archive"]]


,id,subject,jev_category,confidence,jev_action,clear,final,label,archive
0,demo-01,Checkout is down for all customers,urgent,1.00,label,0.91,urgent,Jev/Urgent,False
1,demo-02,Final notice: your server will be suspended in...,urgent,0.93,label,0.91,urgent,Jev/Urgent,False
2,demo-03,Need the signed contract before 5pm today,urgent,1.00,label,0.88,urgent,Jev/Urgent,False
3,demo-04,"Water leak in your flat, access needed this af...",urgent,1.00,label,0.93,urgent,Jev/Urgent,False
4,demo-05,Dinner next month?,reply_later,1.00,label,0.92,reply_later,Jev/Reply Later,False
5,demo-06,Feedback on the onboarding doc,reply_later,1.00,label,0.86,reply_later,Jev/Reply Later,False
6,demo-07,Intro chat about a role?,reply_later,0.93,label,0.57,reply_later,Jev/Reply Later,False
7,demo-08,Question about next quarter's plan,reply_later,1.00,label,0.81,reply_later,Jev/Reply Later,False
8,demo-09,This week in AI: 7 things you missed,newsletter,1.00,archive,0.90,newsletter,Jev/Newsletter,True
9,demo-10,20% off everything this weekend,newsletter,0.99,archive,0.91,newsletter,Jev/Newsletter,True


## How did Jev do? (real numbers from this run)

We compare Jev's final route with the hand labels, and measure speed and tokens. These numbers come from the run you just did, not from a slide.


In [7]:
table["expected"] = [m["expected"] for m in DEMO_INBOX]
table["match"] = table["final"] == table["expected"]
print(f"Matched the hand label: {table['match'].sum()} of {len(table)}")
print("Routes:", table["final"].value_counts().to_dict())
print(f"Sent to Review: {(table['final'] == 'review').sum()}   Archived (planned): {table['archive'].sum()}")
print(f"Jev time per email: median {table['ms'].median():.0f} ms, max {table['ms'].max():.0f} ms")
print(f"Tokens: {table['tok_in'].sum()} in, {table['tok_out'].sum()} out across {len(table)} emails")
misses = table[~table["match"]][["id", "subject", "expected", "final", "confidence", "clear"]]
print("\nDifferent from the hand label:" if len(misses) else "\nNo differences.")
misses


Matched the hand label: 18 of 18
Routes: {'newsletter': 5, 'urgent': 4, 'reply_later': 4, 'spam': 3, 'review': 2}
Sent to Review: 2   Archived (planned): 8
Jev time per email: median 50 ms, max 111 ms
Tokens: 12231 in, 1794 out across 18 emails

No differences.


,id,subject,expected,final,confidence,clear


## Step 4: Apply labels and archive (Composio, or the Gmail API directly)

Composio is here only for easy Gmail access: one OAuth link and no Google Cloud project. Under the hood each action is a single Gmail API call:

| What we want | Composio tool | Plain Gmail API |
|---|---|---|
| list labels | `GMAIL_LIST_LABELS` | `users.labels.list` |
| create a label | `GMAIL_CREATE_LABEL` | `users.labels.create` |
| label / archive | `GMAIL_ADD_LABEL_TO_EMAIL` | `users.messages.modify(addLabelIds, removeLabelIds=["INBOX"])` |

So if you'd rather not use Composio, replace `apply()` below with `service.users().messages().modify(...)` from `google-api-python-client`. Jev and the rest of the notebook stay the same.

With `DRY_RUN=True`, or on the demo inbox, this cell only prints the plan.


In [8]:
def plan_lines(rows):
    for d in rows:
        if d["label"] == "-":
            print(f"  leave            | {d['id']}")
            continue
        verb = "label + ARCHIVE" if d["archive"] else "label"
        print(f"  {verb:16} | {d['label']:16} | {d['id']}")


def apply(composio_tool, message_id, label_id, archive):
    """One Gmail change. Archive = remove the INBOX label. Never deletes."""
    return composio_tool("GMAIL_ADD_LABEL_TO_EMAIL", {
        "user_id": "me",
        "message_id": message_id,
        "add_label_ids": [label_id],
        "remove_label_ids": ["INBOX"] if archive else [],
    })


print("Planned Gmail changes:")
plan_lines(decisions)
if not USE_LIVE_GMAIL or DRY_RUN:
    print("\nDRY RUN / demo inbox: nothing was labeled, archived or deleted.")


Planned Gmail changes:
  label            | Jev/Urgent       | demo-01
  label            | Jev/Urgent       | demo-02
  label            | Jev/Urgent       | demo-03
  label            | Jev/Urgent       | demo-04
  label            | Jev/Reply Later  | demo-05
  label            | Jev/Reply Later  | demo-06
  label            | Jev/Reply Later  | demo-07
  label            | Jev/Reply Later  | demo-08
  label + ARCHIVE  | Jev/Newsletter   | demo-09
  label + ARCHIVE  | Jev/Newsletter   | demo-10
  label + ARCHIVE  | Jev/Newsletter   | demo-11
  label + ARCHIVE  | Jev/Newsletter   | demo-12
  label + ARCHIVE  | Jev/Newsletter   | demo-13
  label + ARCHIVE  | Jev/Spam         | demo-14
  label + ARCHIVE  | Jev/Spam         | demo-15
  label + ARCHIVE  | Jev/Spam         | demo-16
  label            | Jev/Review       | demo-17
  label            | Jev/Review       | demo-18

DRY RUN / demo inbox: nothing was labeled, archived or deleted.


## Optional: run it on your real Gmail

Set `INBOX_LIVE=true` (or `USE_LIVE_GMAIL = True` in the setup cell), add `COMPOSIO_API_KEY`, and run this cell. It connects Gmail once, reads up to `MAX_EMAILS` unread emails, and runs the exact same Jev steps. Senders and subjects are masked in the output. It still only **plans** until you also set `DRY_RUN = False`.


In [9]:
def mask(text):
    return re.sub(r"[A-Za-z0-9]", "•", text or "")[:24]


if not USE_LIVE_GMAIL:
    print("Live Gmail is off (USE_LIVE_GMAIL=False). The demo inbox above is the whole run.")
else:
    from composio import Composio
    composio = Composio(api_key=get_secret("COMPOSIO_API_KEY"), toolkit_versions={"gmail": "latest"})
    USER_ID = os.environ.get("COMPOSIO_USER_ID") or "inbox-assistant"

    def tool(slug, args):
        res = composio.tools.execute(slug, args, user_id=USER_ID, dangerously_skip_version_check=True)
        if not res.get("successful"):
            raise RuntimeError(f"{slug} failed")
        return res.get("data") or {}

    accounts = composio.connected_accounts.list(user_ids=[USER_ID], toolkit_slugs=["gmail"], statuses=["ACTIVE"]).items
    if not accounts:
        cfg = composio.auth_configs.create("gmail", {"type": "use_composio_managed_auth"})
        req = composio.connected_accounts.link(USER_ID, cfg.id)
        print("Open this link to connect Gmail, then re-run this cell:\n", req.redirect_url)
    else:
        data = tool("GMAIL_FETCH_EMAILS", {"user_id": "me", "query": "is:unread in:inbox", "max_results": MAX_EMAILS})
        live = []
        for m in data.get("messages") or []:
            heads = {h["name"].lower(): h["value"] for h in (m.get("payload") or {}).get("headers") or []}
            live.append({"id": m.get("messageId"), "from": m.get("sender") or heads.get("from", ""),
                         "subject": m.get("subject") or "", "snippet": (m.get("messageText") or "")[:300],
                         "list_unsubscribe": "list-unsubscribe" in heads})
        live_decisions = [decide(preprocess(m)) for m in live]
        print(f"Read {len(live)} unread emails (read-only).")
        print(pd.Series([d["final"] for d in live_decisions]).value_counts().to_string())
        for d in live_decisions:
            print(f"  {d['final']:12} conf={d['confidence']:.2f}  subject={mask(d['subject'])}")
        if DRY_RUN:
            print("DRY_RUN=True: no labels created, nothing archived.")
        else:
            existing = {l["name"]: l["id"] for l in tool("GMAIL_LIST_LABELS", {"user_id": "me"}).get("labels", [])}
            for name in {d["label"] for d in live_decisions if d["label"] != "-"} - set(existing):
                existing[name] = tool("GMAIL_CREATE_LABEL", {"user_id": "me", "label_name": name}).get("id")
            for d in live_decisions:
                if d["label"] != "-":
                    apply(tool, d["id"], existing[d["label"]], d["archive"])
            print("Labels applied. Archived emails are still in All Mail. Nothing was deleted.")


Live Gmail is off (USE_LIVE_GMAIL=False). The demo inbox above is the whole run.


## Recap

- **Python** read the envelope: domain, unsubscribe header, contact, deadline words.
- **Jev** made every decision: category, action and "clear enough?", each with a confidence. This is the part that replaces the LLM, and it never wrote a word.
- A **confidence gate** sent unsure emails to *Review* instead of guessing.
- **Composio** (or the plain Gmail API) applies labels and archives, only when `DRY_RUN=False`. Nothing is ever deleted.

LLMs generate, Jev decides.
